# Tweety-11 — Raisonnement Causal : du do-calculus aux contrefactuels

| | |
|---|---|
| **Série** | [TweetyProject](README.md) — Notebook 11 (Synthèse) |
| **Précédent** | [Tweety-10-MLN-Python](Tweety-10-MLN-Python.ipynb) (Markov Logic Networks) |
| **Suivant** | — (fin de la série) |
| **Module Tweety** | `org.tweetyproject.causal` (causal-1.30.jar) |
| **Durée estimée** | ~50 min |
| **Niveau** | Avancé (capstone) |

## Objectifs d'apprentissage

À l'issue de ce notebook, vous saurez :

1. **Distinguer corrélation et causalité** au sein d'un moteur formel — pas seulement intuitivement.
2. **Construire un Structural Causal Model (SCM)** : encoder un graphe causal comme des équations structurelles.
3. **Maîtriser l'opérateur do(X)** de Pearl — l'intervention qui *casse* les liens amont — et le distinguer de l'observation.
4. **Formuler des requêtes contrefactuelles** (« si X avait été différent, Y aurait-il eu lieu ? »), le 3ᵉ niveau de l'échelle de Pearl.
5. **Lever le paradoxe des confondeurs** (baromètre, sprinkler) par le raisonnement causal plutôt que probabiliste.

## Prérequis

- **[Tweety-05-Abstract-Argumentation-Python](Tweety-05-Abstract-Argumentation-Python.ipynb) §4.4** : le raisonnement causal par argumentation (couche *observationnelle* / diagnostique).
- **[Tweety-2-Basic-Logics](Tweety-02-Basic-Logics-Python.ipynb)** : logique propositionnelle (syntaxe des formules PL).
- Bases de probabilités (P(y|x), indépendance conditionnelle).

## Positionnement dans la série

Ce notebook est **complémentaire** de Tweety-5 §4.4, **non redondant** :

| Aspect | Tweety-5 §4.4 (observation) | **Tweety-11 (ce notebook)** |
|--------|------------------------------|------------------------------|
| Question typique | « Sachant les *symptômes*, quelle est la *cause* ? » (diagnostic) | « Si j'**agis** sur X, quel effet sur Y ? » (prédiction d'action) |
| Niveau de Pearl | (1) Association / observation | (2) Intervention `do(X)` + (3) Contrefactuel |
| API Tweety | `ArgumentationBasedCausalReasoner.query(observations)` | `InterventionalStatement`, `CounterfactualStatement`, `do(X)` |

Tweety-5 répond à *« que déduire de ce que je vois ? »* ; Tweety-11 répond à *« que se passerait-il si j'agissais différemment ? »*.

## 1. Pourquoi la causalité ? — L'échelle de Pearl

La logique classique (notebooks 1-4) et probabiliste (Tweety-10 MLN) répondent à la question **« que peut-on déduire d'un ensemble de croyances ? »**. Mais elles sont **aveugles à la direction causale** : un théorème de Bayes ne distingue pas « le baromètre baisse → orage » de « l'orage → le baromètre baisse ».

**Judea Pearl** (Prix Turing 2011) a formalisé cette distinction via l'**échelle causale** à trois niveaux :

| Niveau | Question | Symbole | Exemple |
|--------|----------|---------|---------|
| **1. Association** | Qu'observe-t-on ensemble ? | $P(y \mid x)$ | Les patients sous traitement guérissent plus souvent. |
| **2. Intervention** | Que se passe-t-il si j'**agis** sur X ? | $P(y \mid do(x))$ | Si je **prescris** le traitement, la guérison augmente-t-elle ? |
| **3. Contrefactuel** | Que serait-il arrivé **si** X avait été différent ? | $P(y_x \mid x', y')$ | Ce patient aurait-il guéri **sans** traitement ? |

Le saut du niveau 1 au niveau 2 est **révolutionnaire** : $P(y \mid x) \neq P(y \mid do(x))$ en présence d'un **confondeur** (cause commune). C'est la frontière entre *corrélation* et *causalité*, inaccessible à la statistique seule.

> **L'idée centrale du do-calculus** : `do(X = x)` ne signifie pas « observer X = x » mais **« forcer X à valoir x en remplaçant son équation causale par une constante »**. Cette opération *rompt* tous les liens entrants vers X, isolant l'effet causal propre de X sur le reste du système.

### 1.1 Structural Causal Model (SCM)

Un SCM encode un graphe causal comme un système d'**équations structurelles** :

$$X_i = f_i(\text{Parents}(X_i), U_i)$$

où $U_i$ est un bruit exogène (les *atomes de fond* chez Tweety). Dans ce notebook, nous travaillons en logique propositionnelle déterministe : chaque atome *explicable* est défini comme une formule PL de ses parents, et chaque atome *de fond* est une cause racine (non expliquée dans le modèle).

La sémantique est **symbolique** (raisonnement argumentatif sur les mondes possibles), pas numérique — Tweety fait de la causalité *qualitative*, complémentaire de l'approche bayésienne numérique.

## 2. Configuration du backend causal Tweety (règle F)

Ce notebook exige le JAR `causal-1.30.jar` (module `org.tweetyproject.causal`), invoqué via **JPype** (pont Python↔JVM). Aucune simulation, aucun stub : toutes les requêtes ci-dessous sont exécutées par le vrai moteur causal de Tweety.

**Règle F (fail-loud)** : si l'API causale n'est pas invocable, le notebook **échoue bruyamment** plutôt que de produire une sortie de substitution.

In [13]:
import os, sys

TWEETY_DIR = os.path.abspath(".")   # ce notebook vit dans le dossier Tweety/
assert os.path.isfile(os.path.join(TWEETY_DIR, "tweety_init.py")), \
    "Lancez ce notebook depuis le dossier Tweety/ (tweety_init.py introuvable)"
os.chdir(TWEETY_DIR)
sys.path.insert(0, os.getcwd())

from tweety_init import init_tweety
ok = init_tweety(verbose=False)
print("Backend Tweety initialise :", ok)

Backend Tweety initialise : True


In [14]:
# Verification fail-loud : le module causal est-il reellement present ?
import glob
causal_jars = [j for j in glob.glob(os.path.join(TWEETY_DIR, "libs", "*.jar"))
               if "causal" in os.path.basename(j).lower()]
assert causal_jars, "causal-1.30.jar MANQUANT dans libs/ -- regle F : installer, ne pas contourner"
print("JAR causal detecte :", [os.path.basename(j) for j in causal_jars])

# Imports de l'API causale (verifies au prealable via reflection)
from org.tweetyproject.logics.pl.syntax import Proposition, Negation
from org.tweetyproject.logics.pl.parser import PlParser
from org.tweetyproject.causal.syntax import StructuralCausalModel, CausalKnowledgeBase
from org.tweetyproject.causal.semantics import InterventionalStatement, CounterfactualStatement
from org.tweetyproject.causal.reasoner import ArgumentationBasedCausalReasoner, ArgumentationBasedCounterfactualReasoner
print("Imports causaux OK : StructuralCausalModel, CausalKnowledgeBase, "
      "InterventionalStatement, CounterfactualStatement, 2 reasoners")

JAR causal detecte : ['causal-1.30.jar']
Imports causaux OK : StructuralCausalModel, CausalKnowledgeBase, InterventionalStatement, CounterfactualStatement, 2 reasoners


**Carte de l'API causale** (vérifiée par réflexion sur `causal-1.30.jar`) :

| Classe | Rôle |
|--------|------|
| `StructuralCausalModel` | Le graphe causal : atome de fond (cause racine) + atome explicable (équation) |
| `CausalKnowledgeBase` | SCM + hypothèses sur les atomes de fond (nécessaire pour les contrefactuels) |
| `ArgumentationBasedCausalReasoner` | Requêtes observation + intervention (renvoie un `boolean` qualitatif) |
| `InterventionalStatement` | Encode une requête `do(X)` |
| `CounterfactualStatement` | Encode un contrefactuel « si X avait été différent » |
| `ArgumentationBasedCounterfactualReasoner` | Requêtes contrefactuelles (via twin model) |

> **Note API** : un atome explicable a **exactement une** équation causale. Pour exprimer « $X$ arrive si $A$ **ou** $B$ », on passe une formule composée `A || B`, pas deux appels séparés (le second lèverait `Atom already exists in the model`).

## 3. Exemple 1 — Le baromètre de Pearl (confondeur à 3 nœuds)

Scénario canonique de la littérature causale. Trois propositions :

- **`storm`** (orage) — cause racine, atome de fond.
- **`drops`** (le baromètre baisse) — expliqué par `storm`.
- **`rain`** (il pleut) — expliqué par `storm`.

Le graphe causal est donc `storm → drops` et `storm → rain` : **`storm` est un confondeur** de `drops` et `rain`. Intuitivement, observer que le baromètre baisse *prédit* la pluie (corrélation), mais **forcer** le baromètre à baisser ne fait *pas* pleuvoir (pas de causalité).

In [15]:
plp = PlParser()
def F(s):
    """Raccourci : parse une formule PL depuis une chaine."""
    return plp.parseFormula(s)

storm  = Proposition("storm")
drops  = Proposition("drops")
rain   = Proposition("rain")

barometer = StructuralCausalModel()
barometer.addBackgroundAtom(storm)            # storm = cause racine exogene
barometer.addExplainableAtom(drops, storm)    # drops  <=> storm
barometer.addExplainableAtom(rain,  storm)    # rain   <=> storm

print("SCM barometre :")
print(barometer.prettyPrint())

SCM barometre :
Background atoms: [storm]
Structural Equations:
drops <=> storm
rain <=> storm



Le graphe causal : `storm → {drops, rain}`. La structure `drops <=> storm` et `rain <=> storm` encode les deux flèches sortant de `storm`.

### 3.1 Niveau 1 — Observation (corrélation)

*Sachant que le baromètre baisse, pleut-il ?*

In [16]:
# Base de connaissance causale : on couvre les 2 mondes possibles pour l'atome de fond
ckb_bar = CausalKnowledgeBase(barometer)
ckb_bar.addAssumption(storm)
ckb_bar.addAssumption(Negation(storm))

obs_reasoner = ArgumentationBasedCausalReasoner()

# NIVEAU 1 : observation (correlation)
p_rain_if_drops = obs_reasoner.query(ckb_bar, [drops], rain)
print("[OBSERVATION] observe(drops=True) -> pleut-il ?", p_rain_if_drops)
print("  -> Le barometre predit la pluie : P(rain | drops) = True (correlation via storm)")

[OBSERVATION] observe(drops=True) -> pleut-il ? True
  -> Le barometre predit la pluie : P(rain | drops) = True (correlation via storm)


**Lecture** : `observe(drops) → rain = True`. Statistiquement correct — le baromètre baisse et la pluie sont corrélés via l'orage. Mais cela *n'établit aucune causalité* du baromètre vers la pluie.

### 3.2 Niveau 2 — Intervention `do(drops)` (causalité)

*Si je **force** le baromètre à baisser (je le débranche et je le règle à la main), pleut-il ?*

In [17]:
# NIVEAU 2 : intervention do(drops=True)
do_stmt = InterventionalStatement()
do_stmt.addIntervention(drops, True)     # do(drops) = True
do_stmt.setConclusion(rain)
p_rain_if_do_drops = obs_reasoner.query(ckb_bar, do_stmt)
print("[INTERVENTION] do(drops=True)     -> pleut-il ?", p_rain_if_do_drops)
print("  -> Forcer le barometre NE FAIT PAS pleuvoir : P(rain | do(drops)) = False")

[INTERVENTION] do(drops=True)     -> pleut-il ? False
  -> Forcer le barometre NE FAIT PAS pleuvoir : P(rain | do(drops)) = False


**Lecture** : `do(drops) → rain = False`. C'est la **signature du do-calculus** :

$$\boxed{P(\text{rain} \mid \text{drops}) = \text{True} \quad \neq \quad P(\text{rain} \mid do(\text{drops})) = \text{False}}$$

L'intervention `do(drops)` a **remplacé l'équation** `drops <=> storm` par la constante `True`, **rompant** le lien `storm → drops`. Il ne reste alors aucun chemin de `drops` vers `rain`. C'est exactement ce que la statistique seule ne sait pas exprimer.

### 3.3 Visualiser l'opérateur `do`

La méthode `intervene(prop, bool)` renvoie un **nouveau SCM** où l'équation de `prop` est remplacée par une constante. Comparons avant/après :

In [18]:
print("SCM original :")
print("  ", barometer.prettyPrint().replace("\n", "\n   "))

print("\nSCM apres do(drops=True) :")
barometer_do = barometer.intervene(drops, True)
print("  ", barometer_do.prettyPrint().replace("\n", "\n   "))
print("\n=> L'equation 'drops <=> storm' est devenue 'drops <=> +' (constante).")
print("   Le lien storm -> drops est ROMPU : drops ne depend plus de storm.")

SCM original :
   Background atoms: [storm]
   Structural Equations:
   drops <=> storm
   rain <=> storm
   

SCM apres do(drops=True) :
   Background atoms: [storm]
   Structural Equations:
   drops <=> +
   rain <=> storm
   

=> L'equation 'drops <=> storm' est devenue 'drops <=> +' (constante).
   Le lien storm -> drops est ROMPU : drops ne depend plus de storm.


## 4. Exemple 2 — Le réseau Sprinkler (Pearl, 4 nœuds, chemins multiples)

Le scénario le plus enseigné en causalité (Pearl, *Causality* 2009). Quatre propositions :

- **`cloudy`** (nuageux) — cause racine.
- **`sprinkler`** (arrosage activé) — expliqué par `cloudy`.
- **`rain`** (pluie) — expliqué par `cloudy`.
- **`wet`** (herbe mouillée) — expliqué par `sprinkler OR rain` (conjonction de causes).

Graphe : `cloudy → sprinkler`, `cloudy → rain`, `{sprinkler, rain} → wet`.

Ici, `sprinkler` et `rain` sont **corrélés** (via `cloudy`) sans que l'un cause l'autre. C'est le piège classique : observer qu'il pleut « prédit » que l'arrosage est actif, mais **forcer la pluie** n'active pas l'arrosage.

In [19]:
cloudy    = Proposition("cloudy")
sprinkler = Proposition("sprinkler")
rain2     = Proposition("rain")
wet       = Proposition("wet")

sprinkler_net = StructuralCausalModel()
sprinkler_net.addBackgroundAtom(cloudy)
sprinkler_net.addExplainableAtom(sprinkler, cloudy)
sprinkler_net.addExplainableAtom(rain2, cloudy)
sprinkler_net.addExplainableAtom(wet, F("sprinkler || rain"))   # wet = sprinkler OR rain

print("SCM reseau Sprinkler :")
print(sprinkler_net.prettyPrint())

ckb_sp = CausalKnowledgeBase(sprinkler_net)
ckb_sp.addAssumption(cloudy)
ckb_sp.addAssumption(Negation(cloudy))
print("Base causale prete (assomptions sur cloudy : les 2 mondes).")

SCM reseau Sprinkler :
Background atoms: [cloudy]
Structural Equations:
sprinkler <=> cloudy
wet <=> sprinkler||rain
rain <=> cloudy

Base causale prete (assomptions sur cloudy : les 2 mondes).


### 4.1 Observation vs Intervention — le paradoxe levé

Comparons `observe(rain)` et `do(rain)` sur leur effet sur `sprinkler` :

In [20]:
print("=== Correlation vs Causalite dans le reseau Sprinkler ===\n")

# (a) OBSERVATION : observe(rain=True) -> sprinkler ?
p_sprinkler_if_rain = obs_reasoner.query(ckb_sp, [rain2], sprinkler)
print(f"[OBS]  observe(rain=True)  -> sprinkler actif ? {p_sprinkler_if_rain}")
print("       (correlation : rain et sprinkler coincident via cloudy)\n")

# (b) INTERVENTION : do(rain=True) -> sprinkler ?
do_rain = InterventionalStatement()
do_rain.addIntervention(rain2, True)
do_rain.setConclusion(sprinkler)
p_sprinkler_if_do_rain = obs_reasoner.query(ckb_sp, do_rain)
print(f"[INT]  do(rain=True)       -> sprinkler actif ? {p_sprinkler_if_do_rain}")
print("       (intervention : do(rain) CASSE cloudy->rain, donc plus de correlation avec sprinkler)\n")

# (c) INTERVENTION causale directe : do(sprinkler=True) -> wet ?
do_sprinkler = InterventionalStatement()
do_sprinkler.addIntervention(sprinkler, True)
do_sprinkler.setConclusion(wet)
p_wet_if_do_sprinkler = obs_reasoner.query(ckb_sp, do_sprinkler)
print(f"[INT]  do(sprinkler=True)  -> herbe mouillee ? {p_wet_if_do_sprinkler}")
print("       (causalite directe preservee : sprinkler CAUSE wet)\n")

print("=> P(sprinkler | observe rain) != P(sprinkler | do rain)")
print("   Signature du do-calculus : l'intervention isole l'effet causal propre.")

=== Correlation vs Causalite dans le reseau Sprinkler ===

[OBS]  observe(rain=True)  -> sprinkler actif ? True
       (correlation : rain et sprinkler coincident via cloudy)

[INT]  do(rain=True)       -> sprinkler actif ? False
       (intervention : do(rain) CASSE cloudy->rain, donc plus de correlation avec sprinkler)

[INT]  do(sprinkler=True)  -> herbe mouillee ? True
       (causalite directe preservee : sprinkler CAUSE wet)

=> P(sprinkler | observe rain) != P(sprinkler | do rain)
   Signature du do-calculus : l'intervention isole l'effet causal propre.


**Lecture comparée** :

| Requête | Résultat | Interprétation |
|---------|----------|----------------|
| `observe(rain=True) → sprinkler` | `True` | Corrélation : pluie et arrosage coïncident (cloudy cause les deux) |
| `do(rain=True) → sprinkler` | `False` | **L'intervention casse le chemin** : forcer la pluie n'active pas l'arrosage |
| `do(sprinkler=True) → wet` | `True` | Causalité directe préservée : l'arrosage mouille l'herbe |

C'est exactement la distinction qui permet de répondre à des questions d'action : *« faut-il activer l'arrosage pour compenser le manque de pluie ? »* — la statistique ne sait pas, le SCM oui.

### 4.2 Niveau 3 — Requête contrefactuelle

Le 3ᵉ niveau de Pearl : *« sachant que l'herbe est mouillée, **si l'arrosage avait été désactivé**, l'herbe aurait-elle été mouillée quand même ? »*

Cela requiert un **twin model** — un monde contrefactuel où l'intervention contrefactuelle est appliquée tout en conservant les observations du monde réel. Tweety l'encode via `CounterfactualStatement` + `ArgumentationBasedCounterfactualReasoner`.

In [21]:
cf_reasoner = ArgumentationBasedCounterfactualReasoner()

# CONTREFACTUEL : observe(wet=True) ; si sprinkler avait ete False, wet aurait-il eu lieu ?
cf = CounterfactualStatement()
cf.addObservation(wet)                        # monde reel : on a observe wet=True
cf.addCounterfactualIntervention(sprinkler, False)  # monde contrefactuel : sprinkler=False
cf.setConclusion(wet)                         # question : wet dans le monde contrefactuel ?
result_cf = cf_reasoner.query(ckb_sp, cf)
print(f"[CONTREFACTUEL] observe(wet=True) ; 'si sprinkler avait ete OFF, herbe mouillee ?' {result_cf}")
print("  -> Oui : sous cloudy=True, rain aurait mouille l'herbe meme sans arrosage.")

[CONTREFACTUEL] observe(wet=True) ; 'si sprinkler avait ete OFF, herbe mouillee ?' True
  -> Oui : sous cloudy=True, rain aurait mouille l'herbe meme sans arrosage.


**Lecture** : le contrefactuel répond `True` — sachant que l'herbe est mouillée (et donc probablement `cloudy=True`), si l'arrosage avait été coupé, il aurait **quand même plu**, et l'herbe aurait été mouillée. C'est le raisonnement contrefactuel qui sous-tend la **responsabilité causale** et l'attribution de blâme en droit ou en médecine.

> **Contrainte API** : `CounterfactualStatement` exige au moins une `addAssumption` par atome de fond (ici `cloudy`), faute de quoi le reasoner lève `IllegalArgumentException: There must be at least one assumption for each background atom`. Nous avons couvert les deux mondes (`cloudy` et `¬cloudy`) pour énumérer l'incertitude exogène.

## 5. Exercices

Les trois exercices suivent la convention C.1 (stubs sans erreur volontaire). Complétez-les en vous inspirant des exemples ci-dessus ; le notebook doit rester exécutable de bout en bout.

### Exercice 1 — Construire un SCM (éducation et revenu)

Construisez un SCM modélisant : `ability` (aptitude, atome de fond) cause à la fois `education` (niveau d'études) et `income` (revenu). Autrement dit, `ability` est un **confondeur**.

1. Définissez les propositions et le SCM (3 atomes).
2. Ajoutez les équations structurelles.
3. Vérifiez que `observe(education=True)` prédit `income=True`, mais que `do(education=True)` ne le fait plus (en booléen : `False`).

In [22]:
# Exercice 1 : SCM education / revenu avec le confondeur `ability`
ability   = Proposition("ability")
education = Proposition("education")
income    = Proposition("income")

# 1-2. Le SCM : ability -> education  et  ability -> income
#      (il n'y a AUCUNE fleche education -> income : toute correlation passe par ability)
edu_model = StructuralCausalModel()
edu_model.addBackgroundAtom(ability)              # cause racine exogene = confondeur
edu_model.addExplainableAtom(education, ability)  # education <=> ability
edu_model.addExplainableAtom(income, ability)     # income    <=> ability
print("SCM education/revenu :")
print(edu_model.prettyPrint())

# Base causale : une assumption par valeur de l'atome de fond -> 2 mondes possibles
ckb_edu = CausalKnowledgeBase(edu_model)
ckb_edu.addAssumption(ability)
ckb_edu.addAssumption(Negation(ability))

# 3a. NIVEAU 1 (association) : observe(education=True) -> income ?
result_observe = obs_reasoner.query(ckb_edu, [education], income)

# 3b. NIVEAU 2 (intervention) : do(education=True) -> income ?
do_edu = InterventionalStatement()
do_edu.addIntervention(education, True)   # 'education <=> ability' devient 'education <=> +'
do_edu.setConclusion(income)
result_do = obs_reasoner.query(ckb_edu, do_edu)

# Controle : do(education) n'entraine pas non plus NON(income) -> income est INDETERMINE, pas "faux"
do_edu_neg = InterventionalStatement()
do_edu_neg.addIntervention(education, True)
do_edu_neg.setConclusion(Negation(income))
result_do_neg = obs_reasoner.query(ckb_edu, do_edu_neg)

print(f"[OBS] observe(education=True) -> income ?     {result_observe}")
print(f"[INT] do(education=True)      -> income ?     {result_do}")
print(f"[INT] do(education=True)      -> NON income ? {result_do_neg}   (ni l'un ni l'autre : indetermine)")
print("\nSCM apres do(education=True) :")
print(edu_model.intervene(education, True).prettyPrint())

# Regle F : on echoue bruyamment si le moteur ne donne pas le resultat attendu
assert result_observe and not result_do and not result_do_neg, "Exercice 1 : resultat inattendu"

SCM education/revenu :
Background atoms: [ability]
Structural Equations:
education <=> ability
income <=> ability

[OBS] observe(education=True) -> income ?     True
[INT] do(education=True)      -> income ?     False
[INT] do(education=True)      -> NON income ? False   (ni l'un ni l'autre : indetermine)

SCM apres do(education=True) :
Background atoms: [ability]
Structural Equations:
education <=> +
income <=> ability



**Lecture (Exercice 1)** — énumération des mondes possibles (un par valeur de `ability`) :

| Monde | `ability` | `education` | `income` |
|---|---|---|---|
| W1 | ⊤ | ⊤ | ⊤ |
| W2 | ⊥ | ⊥ | ⊥ |

- `observe(education)` élimine W2 (incompatible avec `education=⊤`) → dans le seul monde restant, `income=⊤` → **True** (raisonnement *abductif* : education ⇒ ability ⇒ income).
- `do(education)` remplace l'équation par `education <=> +` : W2 devient (`ability=⊥`, `education=⊤`, `income=⊥`). Aucune observation n'élimine plus W2, `income` varie d'un monde à l'autre → **False** (non entraîné sceptiquement). Le contrôle `do(education) → ¬income = False` montre que la réponse signifie « indéterminé », pas « le revenu baisse ».

### Exercice 2 — Prédire l'effet d'une intervention

Dans le réseau Sprinkler (`sprinkler_net` déjà construit), prédisez **puis vérifiez** :

1. `do(cloudy=True) → wet` ? (intervention sur la cause racine)
2. `do(wet=True) → rain` ? (intervention sur un effet — que se passe-t-il ?)

Réfléchissez au résultat attendu **avant** d'exécuter, puis confirmez avec le reasoner.

In [23]:
# Exercice 2 : predire puis verifier deux interventions sur sprinkler_net
# Predictions faites AVANT execution :
#   (a) do(cloudy=True) -> wet  : True  (cloudy -> rain -> wet, et cloudy -> sprinkler -> wet)
#   (b) do(wet=True)    -> rain : False (indetermine : forcer un effet ne remonte pas vers ses causes)
from java.lang import IllegalArgumentException
from java.util import HashSet, HashMap

# ---------- (a) do(cloudy=True) -> wet ----------
# Tentative directe : Tweety REFUSE d'intervenir sur un atome de FOND (seuls les atomes explicables
# ont une equation a remplacer). On le montre sans casser l'execution du notebook.
do_cloudy = InterventionalStatement()
do_cloudy.addIntervention(cloudy, True)
do_cloudy.setConclusion(wet)
try:
    obs_reasoner.query(ckb_sp, do_cloudy)
except IllegalArgumentException as e:
    print("[INT] do(cloudy=True) direct -> refuse par Tweety :", e.getMessage())

# Voie 1 : cloudy n'a AUCUN parent, donc aucun chemin "porte derobee" (back-door) :
#          do(cloudy) = observe(cloudy)  (regle 2 du do-calculus)
ex2_a = obs_reasoner.query(ckb_sp, [cloudy], wet)

# Voie 2 : fixer le contexte exogene -> une base causale avec la seule assumption 'cloudy'
ckb_sp_cloudy = CausalKnowledgeBase(sprinkler_net)
ckb_sp_cloudy.addAssumption(cloudy)
ex2_a_bis = bool(obs_reasoner.query(ckb_sp_cloudy, wet))

print(f"[INT] do(cloudy=True) (= observe, cause racine) -> wet ? {ex2_a}")
print(f"[INT] contexte exogene fixe a cloudy=True       -> wet ? {ex2_a_bis}\n")

# ---------- (b) do(wet=True) -> rain ----------
do_wet = InterventionalStatement()
do_wet.addIntervention(wet, True)      # 'wet <=> sprinkler||rain' devient 'wet <=> +'
do_wet.setConclusion(rain2)
ex2_b = obs_reasoner.query(ckb_sp, do_wet)

do_wet_neg = InterventionalStatement()
do_wet_neg.addIntervention(wet, True)
do_wet_neg.setConclusion(Negation(rain2))
ex2_b_neg = obs_reasoner.query(ckb_sp, do_wet_neg)

# Contraste niveau 1 : OBSERVER l'herbe mouillee, elle, renseigne sur la pluie
ex2_b_obs = obs_reasoner.query(ckb_sp, [wet], rain2)

print(f"[INT] do(wet=True)      -> rain ?     {ex2_b}")
print(f"[INT] do(wet=True)      -> NON rain ? {ex2_b_neg}   (indetermine)")
print(f"[OBS] observe(wet=True) -> rain ?     {ex2_b_obs}   (abduction : wet => sprinkler|rain => cloudy => rain)")

# Les mondes (extensions stables) apres do(wet=True) : rain varie d'un monde a l'autre
interv = HashMap()
interv.put(wet, True)
print("      mondes apres do(wet=True) :", obs_reasoner.getModels(ckb_sp, HashSet(), interv))

assert ex2_a and ex2_a_bis and not ex2_b and not ex2_b_neg and ex2_b_obs, "Exercice 2 : resultat inattendu"

[INT] do(cloudy=True) direct -> refuse par Tweety : The specified variable has to be an explainable atom of the causal model
[INT] do(cloudy=True) (= observe, cause racine) -> wet ? True
[INT] contexte exogene fixe a cloudy=True       -> wet ? True

[INT] do(wet=True)      -> rain ?     False
[INT] do(wet=True)      -> NON rain ? False   (indetermine)
[OBS] observe(wet=True) -> rain ?     True   (abduction : wet => sprinkler|rain => cloudy => rain)
      mondes apres do(wet=True) : [[wet], [sprinkler, wet, rain, cloudy]]


**Lecture (Exercice 2)**

| Requête | Résultat | Pourquoi |
|---|---|---|
| `do(cloudy=True)` direct | `IllegalArgumentException` | Tweety n'autorise `intervene` que sur les atomes **explicables** |
| `do(cloudy=True) → wet` (= `observe(cloudy)`) | `True` | Cause racine : pas de parent, pas de back-door, donc $P(y \mid do(x)) = P(y \mid x)$ ; puis `cloudy ⇒ rain ⇒ wet` |
| `do(wet=True) → rain` | `False` | La chirurgie coupe `{sprinkler, rain} → wet` : `wet` ne transmet plus aucune information vers l'amont. Deux mondes subsistent (`cloudy` vrai ou faux) |
| `observe(wet=True) → rain` | `True` | Observer un effet permet de *remonter* abductivement vers la cause commune `cloudy` |

**À retenir** : une intervention se propage uniquement **vers l'aval** du graphe orienté. Forcer un effet (mouiller l'herbe au tuyau) ne dit rien de ses causes ; l'observer, si.

### Exercice 3 — Requête contrefactuelle

Formulez un contrefactuel sur le réseau Sprinkler : *« sachant qu'il pleut (`rain=True`), **si `cloudy` avait été faux**, aurait-il plu quand même ? »*

Construisez un `CounterfactualStatement`, ajoutez l'observation, l'intervention contrefactuelle, et la conclusion. N'oubliez pas les assumptions sur `cloudy`.

In [24]:
# Exercice 3 : contrefactuel "sachant qu'il pleut, si cloudy avait ete faux, aurait-il plu ?"
#
# Deux pieges de l'API (verifies dans le code source de Tweety 1.30) :
#  (1) cloudy est un atome de FOND. Dans le twin model, les atomes de fond sont PARTAGES entre le
#      monde reel et le monde contrefactuel (getCounterfactualCopy(cloudy) == cloudy) : il n'existe
#      pas de 'cloudy*' sur lequel intervenir -> IllegalArgumentException.
#      => On rend cloudy ENDOGENE avec un bruit exogene u_cloudy : cloudy <=> u_cloudy
#         (c'est exactement X_i = f_i(Parents, U_i) du §1.1).
#  (2) La conclusion doit porter sur la COPIE contrefactuelle rain*, pas sur rain : rain est
#      observe dans le monde reel, donc 'rain' serait trivialement True.

# --- Piege (1), demontre sans casser l'execution ---
cf_naif = CounterfactualStatement()
cf_naif.addObservation(rain2)
cf_naif.addCounterfactualIntervention(cloudy, False)
cf_naif.setConclusion(rain2)
try:
    cf_reasoner.query(ckb_sp, cf_naif)
except IllegalArgumentException as e:
    print("[CF] intervention sur l'atome de fond cloudy -> refuse :", e.getMessage(), "\n")

# --- SCM Sprinkler avec bruit exogene explicite ---
u_cloudy = Proposition("u_cloudy")
sprinkler_net_u = StructuralCausalModel()
sprinkler_net_u.addBackgroundAtom(u_cloudy)                     # U_cloudy : contexte exogene
sprinkler_net_u.addExplainableAtom(cloudy, u_cloudy)            # cloudy    <=> u_cloudy
sprinkler_net_u.addExplainableAtom(sprinkler, cloudy)           # sprinkler <=> cloudy
sprinkler_net_u.addExplainableAtom(rain2, cloudy)               # rain      <=> cloudy
sprinkler_net_u.addExplainableAtom(wet, F("sprinkler || rain")) # wet       <=> sprinkler || rain

ckb_sp_u = CausalKnowledgeBase(sprinkler_net_u)
ckb_sp_u.addAssumption(u_cloudy)              # une assumption par valeur de l'atome de fond
ckb_sp_u.addAssumption(Negation(u_cloudy))

print("Twin model (monde reel + copies etoilees du monde contrefactuel) :")
print(sprinkler_net_u.getTwinModel().prettyPrint())

rain_star = sprinkler_net_u.getCounterfactualCopy(rain2)   # Proposition("rain*")

# --- Le contrefactuel ---
cf_ex3 = CounterfactualStatement()
cf_ex3.addObservation(rain2)                         # monde reel : on observe rain = True
cf_ex3.addCounterfactualIntervention(cloudy, False)  # monde contrefactuel : do(cloudy* = False)
cf_ex3.setConclusion(rain_star)                      # question : rain* ?
result_ex3 = cf_reasoner.query(ckb_sp_u, cf_ex3)

cf_ex3_neg = CounterfactualStatement()
cf_ex3_neg.addObservation(rain2)
cf_ex3_neg.addCounterfactualIntervention(cloudy, False)
cf_ex3_neg.setConclusion(Negation(rain_star))
result_ex3_neg = cf_reasoner.query(ckb_sp_u, cf_ex3_neg)

# Piege (2) : conclusion sur le 'rain' FACTUEL -> vrai trivialement (c'est l'observation)
result_piege = cf_reasoner.query(ckb_sp_u, [rain2], cf_ex3.getInterventions(), rain2)

print(f"[CF] observe(rain) ; si cloudy avait ete faux -> rain* ?     {result_ex3}")
print(f"[CF] observe(rain) ; si cloudy avait ete faux -> NON rain* ? {result_ex3_neg}")
print(f"[CF] (piege) conclusion sur 'rain' factuel                -> {result_piege}  (trivial)")
print("  -> Non : sans nuages, il n'aurait pas plu (rain* = False).")

assert (not result_ex3) and result_ex3_neg, "Exercice 3 : resultat inattendu"

[CF] intervention sur l'atome de fond cloudy -> refuse : The specified variable has to be an explainable atom of the causal model 

Twin model (monde reel + copies etoilees du monde contrefactuel) :
Background atoms: [u_cloudy]
Structural Equations:
sprinkler* <=> cloudy*
rain* <=> cloudy*
sprinkler <=> cloudy
cloudy* <=> u_cloudy
wet <=> sprinkler||rain
wet* <=> sprinkler*||rain*
cloudy <=> u_cloudy
rain <=> cloudy

[CF] observe(rain) ; si cloudy avait ete faux -> rain* ?     False
[CF] observe(rain) ; si cloudy avait ete faux -> NON rain* ? True
[CF] (piege) conclusion sur 'rain' factuel                -> True  (trivial)
  -> Non : sans nuages, il n'aurait pas plu (rain* = False).


**Lecture (Exercice 3)** — les 3 étapes de Pearl (*abduction → action → prédiction*) dans le twin model :

1. **Abduction** (monde réel) : `rain` observé ⇒ `cloudy` (car `rain <=> cloudy`) ⇒ `u_cloudy` (car `cloudy <=> u_cloudy`). Seule l'assumption `u_cloudy` reste cohérente.
2. **Action** (monde contrefactuel) : `do(cloudy* = False)` remplace `cloudy* <=> u_cloudy` par `cloudy* <=> -`. Le contexte `u_cloudy` partagé reste vrai.
3. **Prédiction** : `rain* <=> cloudy*` ⇒ `rain* = ⊥`.

Résultat : `rain*` **non entraîné** (`False`) et `¬rain*` **entraîné** (`True`). *Sachant qu'il a plu, s'il n'y avait pas eu de nuages, il n'aurait pas plu* : `cloudy` est une cause **nécessaire** de `rain` dans ce modèle (« but-for cause »).

> Remarque : le même piège (2) concerne l'exemple du §4.2, qui interroge `wet` au lieu de `wet*`. La réponse y est `True` dans les deux cas (`rain*` reste vrai), mais seule la version `wet*` teste réellement le monde contrefactuel.

## 6. Synthèse — L'échelle de Pearl dans Tweety

| Niveau | Question | API Tweety | Opérateur mathématique |
|--------|----------|------------|------------------------|
| **1. Association** | `observe(X) → Y ?` | `reasoner.query(ckb, [X], Y)` | $P(y \mid x)$ |
| **2. Intervention** | `do(X) → Y ?` | `InterventionalStatement` + `addIntervention` | $P(y \mid do(x))$ |
| **3. Contrefactuel** | « si X avait été différent » | `CounterfactualStatement` + `addCounterfactualIntervention` | $P(y_x \mid x', y')$ |

### Ce qu'il faut retenir

1. **`observe(X) ≠ do(X)`** dès qu'il existe un confondeur. La statistique (Tweety-10 MLN, Tweety-5 observation) reste au niveau 1 ; le do-calculus (ce notebook) atteint le niveau 2.
2. **L'opérateur `do` casse les liens amont** : `intervene(prop, bool)` remplace l'équation `prop <=> parents` par une constante, visualisant mécaniquement la coupure causale.
3. **Les contrefactuels** nécessitent un twin model et des assumptions sur les atomes de fond ; ils sous-tendent la responsabilité causale (droit, médecine, éthique).
4. **La causalité qualitative de Tweety** complète l'approche bayésienne numérique : ici, le moteur argumente sur les mondes possibles pour décider Oui/Non, sans paramètres probabilistes.

### Limites honnêtes (borne intrinsèque du paradigme qualitatif)

Le moteur causal de Tweety est **qualitatif** (booléen) : il ne calcule pas de probabilités numériques $P(y \mid do(x))$, mais décide si une conclusion *peut* / *doit* suivre causalement. Pour des estimations quantitatives (effets moyens, intervalles de confiance), il faudrait coupler le SCM à un estimateur bayésien ou à des données observationnelles (front-door / back-door adjustment). C'est une frontière de paradigme, pas un bug — le verdict SOTA est **SOTA-OK** pour le raisonnement qualitatif, avec cette borne explicite.

### Pour aller plus loin
- **[Notebook-pont — du graphe causal au do-calculus](../../Probas/DecisionTheory/Causal-Bridges/CausalBridges-01-Do-Calculus.ipynb)** : l'armature formelle unifiée de Pearl (échelle + 3 règles du do-calculus) exécutée sur l'outil de référence `dowhy`, qui relie cette série à Infer-5, PyMC-5 et ICT-5.

- **Judea Pearl**, *Causality: Models, Reasoning, and Inference* (Cambridge, 2009) — la référence formelle du do-calculus.
- **Pearl & Mackenzie**, *The Book of Why* (2018) — version accessible, l'échelle causale.
- **[Tweety-05-Abstract-Argumentation-Python](Tweety-05-Abstract-Argumentation-Python.ipynb) §4.4** — la couche observationnelle / diagnostique, complémentaire de ce notebook.
- **[Tweety-10-MLN-Python](Tweety-10-MLN-Python.ipynb)** — l'autre pont symbolique/statistique (FOL pondérée), au niveau 1 de l'échelle.

***

*Notebook 11/11 de la série TweetyProject — capstone de synthèse sur le raisonnement causal.*

## 7. Aller plus loin — le même do-calculus, d'autres paradigmes

Tweety traite la causalité en **logique propositionnelle** : `do(X)` y est *qualitatif* — un atome forcé, un SCM réécrit, une réponse vrai/faux. C'est idéal pour *voir* la différence entre `observe` et `do` sans se noyer dans les nombres, mais Tweety **ne chiffre pas** un effet causal (« de combien la probabilité de pluie change-t-elle ? ») ni ne lève quantitativement un paradoxe de Simpson. Les séries probabilistes du dépôt reprennent **exactement ce notebook** avec des distributions :

- **[Infer-5 — Inférence Causale](../../Probas/Infer/Infer-05-Causal-Inference.ipynb)** : le jumeau **distributionnel par message passing**. `P(Y|do(X))` calculé par mutilation de graphe en Infer.NET (inférence par propagation de messages — EP/VMP par défaut, Gibbs aussi disponible), ajustements **backdoor** et **front-door**, paradoxe de Simpson résolu numériquement.
- **[PyMC-5 — Inférence Causale](../../Probas/PyMC/PyMC-05-Causal-Inference.ipynb)** : la version **MCMC**, avec l'opérateur natif `pm.do(model, {X:x})` et le **contrefactuel par abduction** (postérieur sur les variables exogènes).
- **[ICT-5 — Émergence Causale](../../IIT/ICT-Series/ICT-05-CausalEmergence-Python.ipynb)** : le même `do` monte d'un cran — la distribution d'intervention uniforme `p(C)` *est* `do(X_t = x)` appliqué à tout le micro-état, et mesure à **quelle échelle** un système réalise le plus de travail causal (émergence).

> **Vue d'ensemble des quatre paradigmes** : la section « Ponts causaux : le do-calculus de Pearl à travers les paradigmes » du [README IIT](../../IIT/README.md) cartographie comment le symbolique (ce notebook), le bayésien par message passing, le MCMC et la théorie de l'information instancient le même opérateur `do(·)`.